Grundregeln:
- jede python Klasse hat eine übergeordnete Aufgabe
- jede python Klasse hält nur einen data file
- jede python Klasse importiert nur dann andere Klassen, wenn sie nur durch diese Importe als Gesamtes funktionieren (das ist in den meisten Fällen der DataManager). Wenn eine andere Klasse kein inhärenter Teil ist, wird sie nur als Funktions-Input für die Methode nutzbar gemacht.
- kein cleaning! Statt stillem Bereinigen stoppt die Funktion bei Ungereimtheiten und zeigt eine Fehlermeldung an

Zu adressierende Probleme:
- predicate requisite kleiner Teil der RO-Relationen. Reicht der aus?
- nur Abstracts in pubmed. Volltext-Reviews / Metaanalysen oder Bücher besser?
- manche Attribute sind bei fast jeder Entität TRUE. Sollte man diese entfernen? >90% TRUE?
- Scalability probleme vor allem der Normalisierung und fill_nan
- Vektordatenbank statt Pubmed_Abstracts_df besser?




Adressierte Probleme:
- causally_related ist oft falsch verknüpft. Aus der Liste der Prädikate entfernen? Antwort: Entfernt worden
- bis LLMClient().extract_entities werden nur Attribute ausgefüllt, die TRUE sind. Das ist zu ungenau. Von Anfang an müssen Attribute auch False gelabelt werden. Antwort: Extract entities labelt auch Falsche Entitäten
- mehrere Klassen haben durch das LLM denselben Namen bekommen, was verboten ist. Antwort: Prompt ist angepasst worden
- Abstracts enthalten sehr viele Entitäten, die nichts mit Wunden zu tun haben. Antwort: LLM prompt wurde so angepasst, dass nur domänenspezifische Entitäten integriert werden.
- Die formale Konzeptanalyse hält sich nicht an die BFO/OBO-Architektur. Antwort: attributes haben die form predicate: target. präformierte attributes sind die bfo_classes als is_a:bfo_class um die Einteilung in die BFo-Architektur zu erzwingen.
- es kann mehrere synonme Objekte oder Attribute geben. Antwort: objects und attributes werden per LLM zu standardbegriffen gemappt, dann grouping durch hard code
- Klassennamen werden aus Attributen gebildet. Antwort: Semantische Klassennamen kreiert LLM
- zu viele concepts und attribute? unschaffbar für hard code? erweiterter RAM und TPU
- zu viele Anfragen an LLM? Antwort: batching
- zu komplexe Anfragen an LLM? Antwort: Lange Frage besser als lange Antwort! (gemini 3 flash schafft 1mio tokens)
- komplette Nachverfolgbarkeit für selektives Löschen query-weise oder datumsweise bei schlechten einzelnen Ergebnissen: Antwort: meta_ids
- kein überladen des Speicher: Antwort: immer nur 10 dateien eines Präfix im Speicher und purge method des DataManager()
- LLM-Anfragen dauern zu lange, teils nur eine Anfrage pro 2,5 Minuten. Antwort: asyncio parallel prompting


# 1. Einrichtung der Grundbausteine

## 1.2 Installieren der benötigten Bibliotheken

## 1.3 Import der benötigten Bibliotheken

In [28]:
import nest_asyncio
nest_asyncio.apply()



In [29]:
import pandas as pd
import numpy as np
import re
import os
import glob
import seaborn as sns
from pydantic import validate_call, ConfigDict
import matplotlib.pyplot as plt
from scipy.stats import kruskal, chi2_contingency
import statsmodels
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests
#from adjustText import adjust_text
import statsmodels.formula.api as smf

In [30]:
import pymc as pm
import arviz as az
import tabulate
import jax
import time
import json
import google.genai as genai
from concepts import Context
import owlready2
import time
from google.api_core import exceptions
from owlready2 import *
import asyncio
from Bio import Entrez, Medline
from IPython.display import display
import types
from abc import ABC, abstractmethod
import requests
from datetime import datetime
from pathlib import Path
from google.genai import types
from pydantic import BaseModel, Field
import pickle


from google.genai.errors import ServerError

In [31]:
import sys
from pathlib import Path
from dotenv import load_dotenv

# Works whether VS Code runs the notebook from project root or notebooks/
_cwd = Path().resolve()
_project_root = _cwd if (_cwd / 'config').exists() else _cwd.parent
sys.path.insert(0, str(_project_root))
load_dotenv(_project_root / '.env')

from config.settings import (
    BASE_PATH,
    OWL_OUTPUT_FILE,
    BFO_OWL_URL,
    RO_OWL_URL,
    ENTREZ_EMAIL,
    NCBI_API_KEY,
    GOOGLE_API_KEY,
    BATCH_SIZE,
    CACHE_MAX_FILES_PER_PREFIX,
    PUBMED_SEARCH_TERMS,
    ALLOWED_PREDICATES,
)
print(f"Project root: {_project_root}")
print(f"ALLOWED_PREDICATES: {ALLOWED_PREDICATES}")

Project root: /Users/Strubbelig/Library/Mobile Documents/com~apple~CloudDocs/Wound ontology
ALLOWED_PREDICATES: ['is_a', 'part_of', 'has_part', 'has_quality', 'occurs_in', 'has_participant']


# Klassenbibliothek

## class DataManager

In [ ]:
from wound_ontology.data_manager.data_manager import DataManager


### proof of function

In [33]:
data_manager=DataManager()

Directory ready: /Users/Strubbelig/Library/Mobile Documents/com~apple~CloudDocs/Wound ontology/data/cache
Loading latest JSON: meta_ids_2026-04-08_23-00.json


In [34]:
#data_manager.load_latest_json("list_of_ulms_labels")

In [35]:
#data_manager.purge_by_prefix("context_matrix",confirm=True)

## Connectors

### class BaseConnector

In [ ]:
from wound_ontology.connectors.base_connector import BaseConnector


### class OntoConnector

In [ ]:
from wound_ontology.connectors.onto_connector import OntoConnector


#### class BFOConnector

In [ ]:
from wound_ontology.connectors.bfo_connector import BFOConnector


In [39]:
#try_bfo_onto_conn=BFOConnector(data_manager)
#try_bfo_onto_conn.get_data()

In [40]:
#try_bfo_onto_conn.save()

#### class ROConnector

In [ ]:
from wound_ontology.connectors.ro_connector import ROConnector


#### Proof of function

### class ULMSConnector

In [ ]:
from wound_ontology.connectors.umls_connector import UMLSConnector


#### Proof of function

In [43]:
#try_ulms_conn=UMLSConnector(data_manager,"ec3add19-51b3-49d6-866c-4233f3cd5f8a")

In [44]:
#try_ulms_conn.get_data("stab wound")

In [45]:
#try_ulms_conn.save()

### class PubmedConnector

In [ ]:
from wound_ontology.connectors.pubmed_connector import PubmedConnector


#### Proof of function

In [47]:
#try_pub_conn=PubmedConnector(data_manager, "moritz.milewski@master-bids.de")

In [48]:
#try_pub_conn.get_data("wound classification ICNP")

In [49]:
#try_pub_conn.save()

## LLMClient and helper

### pydantic helper classes

In [ ]:
from wound_ontology.llm.schemas import FCAAttribute, Entity, ExtractResponse, NormalizationEntry, NormalizationResponse, FillNanResponse, SemanticNameResponse


### class LLMClient

In [ ]:
from wound_ontology.llm.llm_client import LLMClient


### Proof of function

In [52]:
llm_client=LLMClient(GOOGLE_API_KEY)

In [53]:
task=llm_client.extract_entities("""A wound forms when biological tissues like skin, mucous membranes, and organs are damaged. Different injuries can cause wounds; properly cleaning and dressing the wounds is essential to prevent infections and additional harm.[1][2] The surgical wound classification (SWC) system was initially developed in 1964 by the National Academy of Sciences and the National Research Council. The SWC system was created to represent the bacterial load in a surgical field. The Centers for Disease Control and Prevention (CDC) later refined this system by establishing 4 different classes of wound statuses outlined below.[3] Each class has a postoperative risk of a surgical site infection (SSI) with scores of 1% to 5%, 3% to 11%, 10% to 17%, and more than 27%, respectively. Class 1 wounds are categorized as clean wounds. These types of wounds are not infected, do not exhibit any signs of inflammation, and are typically closed. If drainage is required, a closed draining approach is recommended. It is worth noting that Class 1 wounds do not involve the respiratory, alimentary, genital, or urinary tracts. Examples of clean wounds include an inguinal hernia repair or a thyroidectomy.
#Class 2 wounds are categorized as clean-contaminated, which means they have a low level of contamination. These types of wounds involve entry into the respiratory, alimentary, genital, or urinary tracts but only under controlled circumstances.
#Class 3 wounds are classified as contaminated and typically result from a breach in sterile techniques or leakage from the gastrointestinal tract. Incisions resulting from acute or nonpurulent inflammation are also considered Class 3 wounds.
#Class 4 wounds are considered to be dirty or infected. These injuries usually occur from inadequate treatment of traumatic wounds, gross purulence, and evident infections. When tissues lose vitality, it can lead to Class 4 wounds. This is often caused by surgery or microorganisms found in perforated organs.[3]""", ALLOWED_PREDICATES,[],[])

/var/folders/mx/nh4b04cn5sldd9fv5rcmyk1c0000gp/T/ipykernel_28085/3411648951.py:1: RuntimeWarning: coroutine 'LLMClient.extract_entities' was never awaited
  task=llm_client.extract_entities("""A wound forms when biological tissues like skin, mucous membranes, and organs are damaged. Different injuries can cause wounds; properly cleaning and dressing the wounds is essential to prevent infections and additional harm.[1][2] The surgical wound classification (SWC) system was initially developed in 1964 by the National Academy of Sciences and the National Research Council. The SWC system was created to represent the bacterial load in a surgical field. The Centers for Disease Control and Prevention (CDC) later refined this system by establishing 4 different classes of wound statuses outlined below.[3] Each class has a postoperative risk of a surgical site infection (SSI) with scores of 1% to 5%, 3% to 11%, 10% to 17%, and more than 27%, respectively. Class 1 wounds are categorized as clean w

In [54]:
import asyncio

# Assuming LLMClient is already imported/defined
llm_client = LLMClient(GOOGLE_API_KEY)

async def process_task():
    # Define the semaphore to limit concurrent requests
    sem = asyncio.Semaphore(10)

    text_input = """A wound forms when biological tissues like skin, mucous membranes, and organs are damaged. Different injuries can cause wounds; properly cleaning and dressing the wounds is essential to prevent infections and additional harm.[1][2] The surgical wound classification (SWC) system was initially developed in 1964 by the National Academy of Sciences and the National Research Council. The SWC system was created to represent the bacterial load in a surgical field. The Centers for Disease Control and Prevention (CDC) later refined this system by establishing 4 different classes of wound statuses outlined below.[3] Each class has a postoperative risk of a surgical site infection (SSI) with scores of 1% to 5%, 3% to 11%, 10% to 17%, and more than 27%, respectively. Class 1 wounds are categorized as clean wounds. These types of wounds are not infected, do not exhibit any signs of inflammation, and are typically closed. If drainage is required, a closed draining approach is recommended. It is worth noting that Class 1 wounds do not involve the respiratory, alimentary, genital, or urinary tracts. Examples of clean wounds include an inguinal hernia repair or a thyroidectomy.
#Class 2 wounds are categorized as clean-contaminated, which means they have a low level of contamination. These types of wounds involve entry into the respiratory, alimentary, genital, or urinary tracts but only under controlled circumstances.
#Class 3 wounds are classified as contaminated and typically result from a breach in sterile techniques or leakage from the gastrointestinal tract. Incisions resulting from acute or nonpurulent inflammation are also considered Class 3 wounds.
#Class 4 wounds are considered to be dirty or infected. These injuries usually occur from inadequate treatment of traumatic wounds, gross purulence, and evident infections. When tissues lose vitality, it can lead to Class 4 wounds. This is often caused by surgery or microorganisms found in perforated organs.[3]""" # Your text here
    relations = ALLOWED_PREDICATES

    async with sem:
        # You must 'await' the extraction if it's an async method
        task = await llm_client.extract_entities(text_input, relations, [], [])
        return task

# To run the script:
if __name__ == "__main__":
    result = asyncio.run(process_task())
    print(result)

[Entity(object='Wound', true_attributes=[FCAAttribute(predicate='is_a', target='Tissue Damage'), FCAAttribute(predicate='has_participant', target='Skin'), FCAAttribute(predicate='has_participant', target='Mucous Membrane'), FCAAttribute(predicate='has_participant', target='Organ')], false_attributes=[]), Entity(object='Class 1 Wound', true_attributes=[FCAAttribute(predicate='is_a', target='Wound'), FCAAttribute(predicate='has_quality', target='Clean'), FCAAttribute(predicate='has_quality', target='Closed')], false_attributes=[FCAAttribute(predicate='has_quality', target='Infected'), FCAAttribute(predicate='has_quality', target='Inflamed'), FCAAttribute(predicate='occurs_in', target='Respiratory Tract'), FCAAttribute(predicate='occurs_in', target='Alimentary Tract'), FCAAttribute(predicate='occurs_in', target='Genital Tract'), FCAAttribute(predicate='occurs_in', target='Urinary Tract')]), Entity(object='Class 2 Wound', true_attributes=[FCAAttribute(predicate='is_a', target='Wound'), FCA

## class ExtractionCache

In [ ]:
from wound_ontology.cache.extraction_cache import ExtractionCache


### helper class _Matrix (FCA context matrix)

In [ ]:
from wound_ontology.fca.matrix import _Matrix


## FCA: Matrix and FormalContext

### class FormalContext

In [ ]:
from wound_ontology.fca.formal_context import FormalContext


#### Proof of function

In [58]:
#fc=FormalContext(data_manager, LLMClient(GOOGLE_API_KEY))

In [59]:
#fc.build_from_cache_slice(cache)

## Mappers

### class BaseMapper

In [ ]:
from wound_ontology.mapping.base_mapper import BaseMapper


from wound_ontology.mapping.normalization_mapper import NormalizationMapper


In [61]:
class NormalizationMapper(BaseMapper):
    def __init__(self, mapper_name, data_manager: DataManager,file_name):
        super().__init__(
            mapper_name=mapper_name,
            data_manager=data_manager,
            file_name=file_name
        )

    @property
    def standards(self) -> list[str]:
        """Read-only unique list of the 'clean' ontology terms."""
        return sorted(list(set(self._mapping.values())))

    @property
    def originals(self) -> list[str]:
        """Read-only list of all raw terms we have already processed."""
        return sorted(list(self._mapping.keys()))


    async def build_chunkwise_mapping(self, terms: list[str], llm, batch_size=5):
        """
        Async version: Normalizes medical terms in parallel waves.
        batch_size: How many chunks of 30 to process simultaneously.
        """
        # 1. Filter out already processed terms
        terms_to_map = [t for t in terms if t not in self.originals]

        if not terms_to_map:
            print(f"[{self.name}] All {len(terms)} terms found in cache.")
            return self.mapping

        # 2. Chunking (Groups of 30 is still good for context window quality)
        chunks = [terms_to_map[i:i + 30] for i in range(0, len(terms_to_map), 30)]
        print(f"[{self.name}] Found {len(terms_to_map)} new terms. Processing {len(chunks)} async chunks...")

        # 3. Semaphore to avoid overwhelming the API
        sem = asyncio.Semaphore(batch_size)

        async def process_chunk(chunk, chunk_id):
            async with sem:
                # We pass the current standards so the LLM reuses existing terms
                results = await llm.create_normalization_mapping(chunk, self.standards)
                print(f"[{self.name}] Chunk {chunk_id+1} complete.")
                return results

        # 4. Fire the waves
        tasks = [process_chunk(chunk, i) for i, chunk in enumerate(chunks)]

        # as_completed allows us to update the mapping and save as each chunk finishes
        for f in asyncio.as_completed(tasks):
            batch_results = await f
            # This triggers your @mapping.setter (incremental save to Drive)
            self.mapping = batch_results

        print(f"[{self.name}] Normalization complete. Total terms in cache: {len(self.originals)}")
        return self.mapping

from wound_ontology.mapping.target_mapper import TargetMapper


In [62]:
class TargetMapper(NormalizationMapper):
  def __init__(self, data_manager):
      super().__init__(
            mapper_name="Mapper of the target terms in matrix attributes",
            data_manager=data_manager,
            file_name="target_mapping")


from wound_ontology.mapping.object_mapper import ObjectMapper


In [63]:
class ObjectMapper(NormalizationMapper):
  def __init__(self, data_manager):
      super().__init__(
            mapper_name="Mapper of the matrix object names",
            data_manager=data_manager,
            file_name="object_mapping")

## Hierarchies

### class BaseHierarchy

In [ ]:
from wound_ontology.hierarchies.base_hierarchy import BaseHierarchy


### class ClassHierarchy

In [ ]:
from wound_ontology.hierarchies.class_hierarchy import ClassHierarchy


from wound_ontology.hierarchies.mereological_hierarchy import MereologicalHierarchy


In [ ]:
class MereologicalHierarchy(BaseHierarchy):
    """
    part_of / has_part hierarchy.
      rows   = branch (more specific / part)
      cols   = root   (more general / whole)
      value  = 1.0  →  branch part_of root

    Cache ingestion:
      'part_of: X'  → stored directly  (row=entity, col=X)
      'has_part: X' → inverted          (row=X, col=entity)
    """

    file_name = "mereological_hierarchy"

    @property
    def _direct_predicates(self):
        return {"part_of"}

    @property
    def _inverse_predicates(self):
        return {"has_part"}

    def _add_owl_axiom(self, onto, branch_cls, root_cls):
        from owlready2 import ObjectProperty, TransitiveProperty
        hp = onto.search_one(iri="*has_part")
        if hp is None:
            with onto:
                class has_part(ObjectProperty, TransitiveProperty):
                    pass
            hp = onto.search_one(iri="*has_part")
        # branch part_of root  ≡  root has_part branch
        if hp is not None:
            root_cls.is_a.append(hp.some(branch_cls))


from wound_ontology.fca.concept_analysis import FormalConceptAnalysis


In [64]:
class FormalConceptAnalysis:

    def __init__(self, data_manager,formal_context):
        self.dm=data_manager
        self.file_name="concept_lattice"
        self.fc = formal_context
        self._lattice = self.dm.load_latest_pickle(self.file_name)

    @property
    def semantic_superclass_names(self):

        """
        Internal logic to filter 'Composite Attributes' out of the hierarchy.
        """
        semantic_names=[]
        all_concepts= self.lattice.concepts
        for concept in all_concepts:

            name = getattr(concept, 'name', None)

            if isinstance(name, str):
                # EXCLUSION LOGIC:
                # If it has a colon, it's a medical triple (Attribute Composite)
                # If it starts with '(', it's a raw tuple
                is_composite = ":" in name or name.startswith("(")

                if not is_composite:
                    semantic_names.append(name)

        return list(semantic_names)


    @property
    def lattice(self):
        if self._lattice is None:
            print("Building lattice...")
            self._lattice = self.build_lattice()
        return self._lattice

    @lattice.setter
    def lattice(self, new_lattice):
        self._lattice = new_lattice
        self.save()

    def save(self):
        self.dm.save_as_pickle(self.lattice,self.file_name)

    def build_lattice(self):
        ctx = Context(self.fc.matrix.objects, self.fc.matrix.attributes, self.fc.matrix.bool_tuples)
        self.lattice = ctx.lattice
        return self.lattice

    def build_lattice(self):
        """
        Purely local calculation.
        Note: We don't make this 'async' because 'Context' is a CPU-bound
        library operation, not a network operation.
        """
        print("🏗️  Constructing Formal Concept Lattice from matrix...")
        # Assuming 'Context' comes from a library like 'concepts' or 'fca'
        from concepts import Context

        ctx = Context(
            self.fc.matrix.objects,
            self.fc.matrix.attributes,
            self.fc.matrix.bool_tuples
        )
        self.lattice = ctx.lattice
        return self.lattice

    async def create_superclass_names(self, llm, min_objects=2, batch_size=10):
        """
        Async version: Generates medical superclass names for concepts in parallel.
        """
        if not self.lattice:
            print("⚠️ No lattice found. Build it first.")
            return None

        # 1. Identify valid concepts that need a name
        concepts_to_name = []
        for concept in self.lattice:
            # Filter logic (Skip top/bottom or tiny/noisy concepts)
            if not concept.intent: continue
            if len(concept.extent) < min_objects: continue

            # If concept already has a custom name, we can skip it (optional)
            concepts_to_name.append(concept)

        print(f"🧬 Found {len(concepts_to_name)} concepts requiring medical naming.")

        # 2. The Semaphore (Traffic Control)
        sem = asyncio.Semaphore(batch_size)

        async def name_worker(concept):
            async with sem:
                try:
                    # Trigger the async LLM method
                    # concept.intent is usually a tuple of strings
                    name = await llm.generate_semantic_name(concept.intent,self.semantic_superclass_names)
                    concept.name = name
                    return True
                except Exception as e:
                    print(f"❌ Failed to name concept {concept.intent[:2]}: {e}")
                    return False

        # 3. Fire the Wave
        tasks = [name_worker(c) for c in concepts_to_name]

        # We use gather to ensure all concepts are named before proceeding
        results = await asyncio.gather(*tasks)

        print(f"✅ Named {sum(results)} superclasses.")
        self.save()
        return self.lattice

## class OWL Exporter

In [ ]:
from wound_ontology.export.owl_exporter import OWLExporter


## class Orchestrator

In [ ]:
from wound_ontology.orchestrator import Orchestrator


# Pipeline

In [67]:
pipeline = Orchestrator()
#pipeline.dm.purge_by_prefix("context_matrix",confirm=True)
#pipeline.run_full_pipeline(PUBMED_SEARCH_TERMS[0])
#pipeline.run_extract_to_final_cache()
pipeline.run_cache_to_nan_matrix()
pipeline.run_matrix_nan_fill()
pipeline.run_fca()
pipeline.export_to_owl()

Directory ready: /Users/Strubbelig/Library/Mobile Documents/com~apple~CloudDocs/Wound ontology/data/cache
Loading latest JSON: meta_ids_2026-04-08_23-00.json
Initializing PubMed API...
Loading latest CSV: pubmed_abstracts_2026-04-08_23-00.csv
Connected to PubMed via moritz.milewski@master-bids.de
No CSV found for prefix: context_matrix
Loading latest JSON: extraction_cache_2026-04-08_23-15.json
No Pickle found for prefix: concept_lattice
No JSON found for prefix: target_mapping
No JSON found for prefix: object_mapping
Synchronizing Matrix...
✅ Sparse Sync Successful (1.0=True, 0.0=False, NaN=Unknown).
   - Unique Entities: 328
   - Medical Triples: 661
Creating normalization mappings (Parallel)...
[Mapper of the target terms in matrix attributes] Found 653 new terms. Processing 22 async chunks...
[Mapper of the target terms in matrix attributes] Chunk 9 complete.
[Mapper of the target terms in matrix attributes] Saving updated mapping...
[Mapper of the target terms in matrix attributes

KeyboardInterrupt: 

In [ ]:
display(pipeline.cache.dict)

In [ ]:
display(pipeline.fc.matrix.df)

In [ ]:
def print_lattice_names(fca_instance):
    print(f"--- 🧬 Exploring {len(fca_instance.lattice)} Lattice Concepts ---")

    named_count = 0
    for i, concept in enumerate(fca_instance.lattice):
        # 1. Check if the LLM actually assigned a name
        # We use getattr to avoid AttributeError if the name wasn't set
        name = getattr(concept, 'name', None)

        if name:
            # 2. Print the Name + some metadata (how many medical objects are in it)
            extent_size = len(concept.extent)
            intent_preview = ", ".join(list(concept.intent)[:3]) # Show first 3 attributes

            print(f"[{i}] Name: {name}")
            print(f"    └─ Extent: {extent_size} objects | Intent: {intent_preview}...")
            named_count += 1

        # Optional: Stop after 50 so you don't scroll forever in Colab
        if named_count >= 50:
            print("\n... and 1,800+ more. Use slicing to see others.")
            break

    print(f"\n✅ Displayed {named_count} named medical superclasses.")

# To run it:
print_lattice_names(pipeline.fca)